# 임베딩 모델 후보 비교 (embedding_test_dy)

프로젝트2 임베딩 모델 선정 — bge-m3 / Qwen3-Embedding / bge-m3-ko (Nemotron 확정 후 추가).
동일 코퍼스(494청크)·동일 테스트셋으로 Recall@k·MRR·인코딩 속도 측정.

**런타임 → GPU** 로 설정하고 위에서부터 실행. 모델 ID·설정은 `src/eval_embeddings.py` 의 `MODELS` 에서 관리.

In [ ]:
# 1) 의존성 — dense 비교용. transformers는 Qwen3(4.51+)용, einops는 NV-Embed-v2 원격코드용
!pip -q install -U sentence-transformers transformers einops

In [ ]:
# 2) 저장소 clone (private면 GitHub 토큰 필요). 토큰 입력창이 뜨면 붙여넣기.
import getpass, os
tok = getpass.getpass('GitHub token (public이면 그냥 Enter): ').strip()
url = f'https://{tok}@github.com/likelion-8/kdic-crawler.git' if tok else 'https://github.com/likelion-8/kdic-crawler.git'
!rm -rf kdic-crawler
!git clone -b embedding_test_dy {url}
%cd kdic-crawler

In [ ]:
# 3) GPU 확인
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 4) 비교 실행 — 모델별 로딩·인코딩·평가. 결과표 출력 + data/embedding_eval/dy.json 저장
!python src/eval_embeddings.py

In [ ]:
# 5) 결과 JSON 확인 (팀원 파일과 합쳐 비교)
import json; print(json.dumps(json.load(open('data/embedding_eval/dy.json', encoding='utf-8')), ensure_ascii=False, indent=2))

---
## (선택) NV-Embed-v2 — 별도 런타임에서

NV-Embed-v2의 원격코드는 **transformers 4.42.4**를 요구하는데 위 Qwen3는 **4.51+**가 필요해 한 환경에서 공존 불가.
그래서 NV-Embed만 아래처럼 따로 돌리고 결과를 `dy.json`의 `nv-embed-v2` 행에 병합한다(`--only`).

**실행 순서:** `런타임 → 세션 다시 시작` → 위 1~6 셀은 건너뛰고 **아래 3셀만** 순서대로. (7B라 다운로드~14GB·인코딩 몇 분)

In [ ]:
# NV-Embed 전용 의존성 — 절대 -U 하지 말 것. NV-Embed 원격코드는 transformers 4.42.4 요구
!pip -q install "transformers==4.42.4" "sentence-transformers==3.0.1" einops

In [ ]:
# 저장소 clone (public이면 토큰창 그냥 Enter)
import getpass
tok = getpass.getpass('GitHub token (public이면 그냥 Enter): ').strip()
url = f'https://{tok}@github.com/likelion-8/kdic-crawler.git' if tok else 'https://github.com/likelion-8/kdic-crawler.git'
!rm -rf kdic-crawler
!git clone -b embedding_test_dy {url}
%cd kdic-crawler

In [ ]:
# NV-Embed만 실행 → dy.json의 nv-embed 행에 병합(기존 3종 결과 보존). 7B라 다운로드·인코딩 몇 분
!python src/eval_embeddings.py --only nv-embed-v2
import json; print(json.dumps(json.load(open('data/embedding_eval/dy.json', encoding='utf-8')), ensure_ascii=False, indent=2))